# PathSnap training

Use a Google Colab CPU runtime. Upload the updated project source ZIP and the supplied student data ZIP. The final evaluator partition is reserved for the live demo.

The training cell compares seven classifiers, including a multilayer perceptron (MLP) and XGBoost, then tunes the best model using grouped cross-validation.

In [ ]:
import os
from pathlib import Path
from zipfile import ZipFile
from google.colab import files

root = Path("/content")
os.chdir(root)
required = ("pathsnap-source.zip", "Student folder.zip")
missing = [name for name in required if not (root / name).is_file()]
if missing:
    print("Upload: " + ", ".join(missing))
    files.upload()
missing = [name for name in required if not (root / name).is_file()]
if missing:
    raise ValueError("Missing ZIP files: " + ", ".join(missing))

source = root / "pathsnap-source"
with ZipFile(root / "pathsnap-source.zip") as bundle:
    if b"xgboost==" not in bundle.read("requirements.txt"):
        raise ValueError("This is the older source ZIP. Delete it from Colab Files, rerun this cell, and upload the updated pathsnap-source.zip.")
    bundle.extractall(source)
os.chdir(source)
%pip install -q -r requirements-colab.txt

In [ ]:
import subprocess
import sys

data = "/content/Student folder.zip"
subprocess.run([sys.executable, "-m", "pathsnap.train", "--data", data], check=True)

## Student-test accuracy after route decoding

Mean trip accuracy gives each student-test trip equal weight. Pooled pointwise accuracy gives each GPS observation equal weight, so longer trips contribute more. Both metrics use the same predictions and reference alignment.

The saved output below comes from a local verification of the final Colab classifier with the revised decoder. Decoder settings were selected using a subset of the training partition. The notebook recomputes both metrics after a new training run.

In [ ]:
import json
from pathlib import Path
from statistics import mean

results = json.loads(Path("artifacts/results.json").read_text(encoding="utf-8"))
comparison = results["decoder_comparison"]
trips = comparison["per_trip"]

print("Decoder settings:", results.get("decoder_settings", "original Colab baseline"))
print(f"Student-test trips: {len(trips)}")
print(f"{'Metric':<40} {'Classifier only':>16} {'With HMM/Viterbi':>17}")
print(f"{'Mean trip accuracy (equal trip weight)':<40} "
      f"{mean(t['classifier_accuracy'] for t in trips):>16.2%} "
      f"{mean(t['decoded_accuracy'] for t in trips):>17.2%}")
print(f"{'Pooled pointwise accuracy':<40} "
      f"{comparison['top1_accuracy']:>16.2%} "
      f"{comparison['decoded_accuracy']:>17.2%}")

Decoder settings: {'gap_seconds': 60, 'routing_limit_m': 1500, 'transition_scale_m': 55, 'disconnect_penalty': 12, 'smoothed_heading_weight': 2}
Student-test trips: 10
Metric                                    Classifier only  With HMM/Viterbi
Mean trip accuracy (equal trip weight)             83.99%            91.41%
Pooled pointwise accuracy                          82.08%            90.37%


In [ ]:
subprocess.run([sys.executable, "-m", "pathsnap.visualize", "--data", data], check=True)
from IPython.display import Image, display
display(Image(filename="artifacts/model-comparison.png"))
display(Image(filename="artifacts/student-trip-map.png"))
display(Image(filename="artifacts/decoder-accuracy-by-trip.png"))
display(Image(filename="artifacts/student-trip-route-diagnostics.png"))

In [ ]:
from shutil import make_archive

bundle = make_archive("/content/pathsnap-artifacts", "zip", "artifacts")
files.download(bundle)